In [ ]:
# Import required code for simulating and visualizing example models
from specparam import SpectralModel
from specparam.sim import sim_power_spectrum
from specparam.sim.utils import set_random_seed
from specparam.plts.spectra import plot_spectra
from specparam.plts.annotate import plot_annotated_model

import scipy.io
import numpy as np
import os


# Check fit for individual subject

In [ ]:
# Import data 
sub_num = '021'

data = scipy.io.loadmat(rf'')

psd = data['psd'].flatten()  # may need conversion from cell to numpy array
freqs = data['freqs'].flatten()

print(freqs.shape)
print(psd.shape)

# Initialize a model object
fm = SpectralModel()

# Set the frequency range to fit the model
freq_range = [4, 45]

# Report: fit the model, print the resulting parameters, and plot the reconstruction
fm.report(freqs, psd, freq_range, peak_width_limits=[1, 8], max_n_peaks=6, min_peak_height=0.15)

cfs = fm.get_params('peak', 'CF')

# LOOP OVER ALL SUBJECTS

In [ ]:
import numpy as np
import scipy.io
from pathlib import Path
from specparam import SpectralModel


DATAIN = rf''
DATAOUT = rf''

subjects = scipy.io.loadmat(rf'')
subject_list = subjects['reftep_active_subject_list'].flatten()
subject_list = [s[0] for s in subject_list]

subjects = subject_list[0:29] # TUBINGEN ONLY

freq_range = [4, 45]
mu_band = [8, 13]

# -------------------------
# LOOP OVER SUBJECTS
# -------------------------
subj_ids        = []
mu_cf_primary   = []
mu_cf_all       = []
for subj in subjects:
    subj = str(subj)  # convert numpy.str_ to Python str
    subids = subj.replace('REFTEP_', '')
    sub_num = 'sub-' + subids

    mat_file = os.path.join(DATAIN, subj, f'{sub_num}_rsEEG_PSD.mat')

    if not os.path.exists(mat_file):
        print(f'Skipping {subj}: file not found')
        continue

    data = scipy.io.loadmat(mat_file)

    freqs = data['freqs'].squeeze()
    psd   = data['psd'].squeeze()

    # Safety checks
    if freqs.ndim != 1 or psd.ndim != 1:
        raise ValueError(f'Data shape issue in {subj}')

    # ---- Fit FOOOF ----
    fm = SpectralModel(
        peak_width_limits=[1, 8],
        max_n_peaks=6,
        min_peak_height=0.1
    )

    fm.fit(freqs, psd, freq_range)

    # ---- Extract peaks ----
    peaks = fm.get_params('peak')

    # peaks columns: [CF, PW, BW]
    if peaks.size == 0:
        subj_ids.append(subj)
        mu_cf_primary.append(np.nan)
        mu_cf_all.append([])
        continue

    cfs = peaks[:, 0]
    pws = peaks[:, 1]

    # ---- Select mu-band peaks ----
    mu_mask = (cfs >= mu_band[0]) & (cfs <= mu_band[1])
    mu_cfs  = cfs[mu_mask]
    mu_pws  = pws[mu_mask]

    subj_ids.append(subj)

    if len(mu_cfs) == 0:
        # mu_cf_primary.append(np.nan)
        # mu_cf_all.append([])
        mu_cf_primary.append(10.5)
        mu_cf_all.append([10.5])
    else:
        # pick strongest mu peak
        idx = np.argmax(mu_pws)
        mu_cf_primary.append(mu_cfs[idx])
        mu_cf_all.append(mu_cfs.tolist())

# -------------------------
# SAVE FOR MATLAB
# -------------------------
individual_mu_peak_frequencies = {
    'subjects': np.array(subj_ids, dtype=object),
    'mu_cf_primary': np.array(mu_cf_primary),
    'mu_cf_all': np.array(mu_cf_all, dtype=object)
}

scipy.io.savemat(os.path.join(DATAOUT, 'individual_mu_peak_frequencies.mat'), individual_mu_peak_frequencies)
